# 🌍 Multi-Agent Travel Advisor
### AI-Powered Trip Planner — Agentathon 2026

**Setup (60 seconds):**
1. Click the 🔑 key icon on the left sidebar → **Secrets**
2. Add `OPENAI_API_KEY` → [platform.openai.com](https://platform.openai.com)
3. Add `TAVILY_API_KEY` → [app.tavily.com](https://app.tavily.com)
4. Add `DISCORD_WEBHOOK_URL` → your Discord webhook
5. Run all cells top to bottom ▶▶

---

In [ ]:
# ── CELL 1: Install dependencies ──────────────────────────────
!pip install openai tavily-python gradio requests stripe --quiet


In [ ]:
# ── CELL 2: Load your API keys ────────────────────────────────
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")
os.environ["STRIPE_SECRET_KEY"] = userdata.get("STRIPE_SECRET_KEY")

DISCORD_WEBHOOK_URL = userdata.get("DISCORD_WEBHOOK_URL")

print("✅ Keys loaded")


In [ ]:
# ── CELL 3: Sub-Agent Definitions ─────────────────────────────
import os, json, requests
from openai import OpenAI
from tavily import TavilyClient

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
tavily = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

# ── Tavily tool (shared by all sub-agents) ──
def search_web(query: str) -> str:
    results = tavily.search(query=query, max_results=5)
    output = []
    for r in results.get("results", []):
        output.append(f"Source: {r['url']}\n{r['content'][:400]}")
    return "\n\n---\n\n".join(output) if output else "No results found."

SEARCH_TOOL = [{"type": "function", "function": {
    "name": "search_web",
    "description": "Search the web for real-time travel info: flights, hotels, activities, prices.",
    "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}
}}]

def _run_sub_agent(system_prompt, user_prompt):
    """Run a sub-agent with tool calling support."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]
    while True:
        resp = client.chat.completions.create(
            model="gpt-4o", messages=messages, tools=SEARCH_TOOL, max_tokens=2000
        )
        msg = resp.choices[0].message
        if resp.choices[0].finish_reason == "tool_calls":
            messages.append(msg)
            for tc in msg.tool_calls:
                args = json.loads(tc.function.arguments)
                print(f"  🔧 {tc.function.name}({args.get('query','')})")
                result = search_web(**args)
                messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
        else:
            return msg.content or ""

# ── SUB-AGENTS ──

def flight_agent(destination, dates, budget):
    print("✈️  Flight Agent searching...")
    return _run_sub_agent(
        system_prompt="""You are a flight search specialist. Find the best flight options.
Return ONLY valid JSON: [{"airline":"...","price":..., "departure":"...","arrival":"...","stops":0,"duration":"..."}]
Search for real current options. Stay within the budget.""",
        user_prompt=f"Find flights to {destination} for {dates}, budget ${budget}"
    )

def hotel_agent(destination, dates, budget):
    print("🏨 Hotel Agent searching...")
    return _run_sub_agent(
        system_prompt="""You are a hotel search specialist. Find the best accommodation options.
Return ONLY valid JSON: [{"name":"...","price_per_night":...,"total":...,"stars":...,"location":"...","amenities":["..."]}]
Search for real current options. Stay within the budget.""",
        user_prompt=f"Find hotels in {destination} for {dates}, budget ${budget}"
    )

def activities_agent(destination, dates, interests):
    print("🎯 Activities Agent searching...")
    return _run_sub_agent(
        system_prompt="""You are a travel activities specialist. Find the best things to do.
Return ONLY valid JSON: [{"name":"...","cost":...,"duration":"...","type":"...","description":"..."}]
Match activities to the traveler's interests. Include a mix of free and paid options.""",
        user_prompt=f"Find activities in {destination} for {dates}. Interests: {interests}"
    )

def budget_agent(flight_data, hotel_data, activities_data, total_budget):
    print("💰 Budget Agent analyzing...")
    return _run_sub_agent(
        system_prompt="""You are a travel budget optimizer. Given flight, hotel, and activity options:
1. Pick the best combination that fits within budget
2. If over budget, suggest specific trade-offs
3. Return a structured itinerary with a cost breakdown
Format as a clear day-by-day plan with costs.""",
        user_prompt=f"""Total budget: ${total_budget}
Flights: {flight_data}
Hotels: {hotel_data}
Activities: {activities_data}

Build the best itinerary within budget."""
    )

print("✅ Sub-agents defined")


In [ ]:
# ── CELL 4: Orchestrator ──────────────────────────────────────
import stripe
import uuid
from datetime import datetime

stripe.api_key = os.environ["STRIPE_SECRET_KEY"]

def post_to_discord(message: str) -> str:
    try:
        response = requests.post(DISCORD_WEBHOOK_URL, json={"content": message}, timeout=10)
        return "Posted to Discord ✅" if response.status_code == 204 else f"Discord error: {response.status_code}"
    except Exception as e:
        return f"Failed: {str(e)}"

def book_trip(destination, total_cost, itinerary_summary):
    """Create a Stripe Checkout session for the trip and post confirmation to Discord."""
    print(f"💳 Creating checkout for {destination} — ${total_cost}...")

    # Create Stripe Checkout Session (test mode)
    session = stripe.checkout.Session.create(
        payment_method_types=["card"],
        line_items=[{
            "price_data": {
                "currency": "usd",
                "product_data": {
                    "name": f"Trip to {destination}",
                    "description": itinerary_summary[:500],
                },
                "unit_amount": int(total_cost * 100),  # Stripe uses cents
            },
            "quantity": 1,
        }],
        mode="payment",
        success_url="https://example.com/success",
        cancel_url="https://example.com/cancel",
    )

    # Generate confirmation
    booking_id = f"TRV-{uuid.uuid4().hex[:8].upper()}"
    confirmation = (
        f"🎫 **BOOKING CONFIRMATION**\n"
        f"━━━━━━━━━━━━━━━━━━━━━━━━━━\n"
        f"📋 Booking ID: {booking_id}\n"
        f"🌍 Destination: {destination}\n"
        f"💰 Total: ${total_cost}\n"
        f"📅 Booked: {datetime.now().strftime('%B %d, %Y %I:%M %p')}\n"
        f"━━━━━━━━━━━━━━━━━━━━━━━━━━\n"
        f"{itinerary_summary}\n"
        f"━━━━━━━━━━━━━━━━━━━━━━━━━━\n"
        f"💳 Pay here (test mode): {session.url}\n"
        f"Use card: 4242 4242 4242 4242 | Any future exp | Any CVC"
    )

    # Post to Discord
    post_to_discord(confirmation)

    return confirmation

ORCHESTRATOR_PROMPT = """You are a travel planning orchestrator. You coordinate specialized sub-agents to plan trips.

When a user asks to plan a trip, extract:
- destination, dates, budget, interests/preferences

Then call plan_trip with those details. After receiving the itinerary, present it clearly and conversationally.
If the user wants changes, call plan_trip again with adjusted parameters.
When the user confirms or says "book it", call book_trip to generate a payment link.
NEVER say you have booked anything — instead provide the checkout link and let the user complete payment."""

ORCH_TOOLS = [
    {"type": "function", "function": {
        "name": "plan_trip",
        "description": "Plan a complete trip by coordinating flight, hotel, and activity agents",
        "parameters": {"type": "object", "properties": {
            "destination": {"type": "string", "description": "Travel destination city/country"},
            "dates": {"type": "string", "description": "Travel dates or duration"},
            "budget": {"type": "number", "description": "Total budget in USD"},
            "interests": {"type": "string", "description": "Traveler interests and preferences"}
        }, "required": ["destination", "dates", "budget"]}
    }},
    {"type": "function", "function": {
        "name": "book_trip",
        "description": "Book the trip — creates a Stripe checkout payment link and posts confirmation to Discord. Use when user confirms they want to book.",
        "parameters": {"type": "object", "properties": {
            "destination": {"type": "string", "description": "Trip destination"},
            "total_cost": {"type": "number", "description": "Total trip cost in USD"},
            "itinerary_summary": {"type": "string", "description": "Brief summary of the itinerary"}
        }, "required": ["destination", "total_cost", "itinerary_summary"]}
    }},
    {"type": "function", "function": {
        "name": "post_to_discord",
        "description": "Post a message to Discord",
        "parameters": {"type": "object", "properties": {
            "message": {"type": "string", "description": "The message to post"}
        }, "required": ["message"]}
    }}
]

def plan_trip(destination, dates, budget, interests="sightseeing"):
    """Orchestrate all sub-agents and return a complete itinerary."""
    print(f"\n🌍 Planning trip to {destination} | {dates} | ${budget} budget")
    print("=" * 50)

    flight_budget = int(budget * 0.4)
    hotel_budget = int(budget * 0.3)
    activity_budget = int(budget * 0.3)

    flights = flight_agent(destination, dates, flight_budget)
    hotels = hotel_agent(destination, dates, hotel_budget)
    activities = activities_agent(destination, dates, interests)

    print(f"\n💰 Budget Agent assembling itinerary...")
    print("=" * 50)
    itinerary = budget_agent(flights, hotels, activities, budget)

    return itinerary

ORCH_FUNCTIONS = {"plan_trip": plan_trip, "book_trip": book_trip, "post_to_discord": post_to_discord}

print("✅ Orchestrator ready")


In [ ]:
# ── CELL 5: Agent loop ────────────────────────────────────────

def run_agent(user_message: str, history: list) -> str:
    messages = [{"role": "system", "content": ORCHESTRATOR_PROMPT}]
    for human, assistant in history:
        messages.append({"role": "user", "content": human})
        messages.append({"role": "assistant", "content": assistant})
    messages.append({"role": "user", "content": user_message})

    while True:
        response = client.chat.completions.create(
            model="gpt-4o", messages=messages, tools=ORCH_TOOLS, max_tokens=2000
        )
        message = response.choices[0].message

        if response.choices[0].finish_reason == "tool_calls":
            messages.append(message)
            for tc in message.tool_calls:
                fn_name = tc.function.name
                fn_args = json.loads(tc.function.arguments)
                print(f"🔧 Orchestrator calling: {fn_name}")
                result = ORCH_FUNCTIONS[fn_name](**fn_args)
                messages.append({"role": "tool", "tool_call_id": tc.id, "content": str(result)})
        else:
            return message.content or "No response."

print("✅ Agent ready — try: 'Plan a 3-day trip to Barcelona for $2000, I like hiking and seafood'")


In [ ]:
# ── CELL 6: Launch ────────────────────────────────────────────
# Your public URL appears below. Share it with anyone.
import gradio as gr

demo = gr.ChatInterface(
    fn=run_agent,
    title="🌍 Multi-Agent Travel Advisor",
    description="Plan your dream trip! I coordinate specialized agents for flights, hotels, and activities — all within your budget.",
    examples=[
        "Plan a 3-day trip to Barcelona for $2000, I like hiking and seafood",
        "I want a week in Tokyo, $3000 budget, interested in anime, street food, and temples",
        "Cheap weekend getaway from NYC under $500, surprise me",
    ],
)

demo.launch(share=True)


---
## 🛠️ Stretch Goals

- **Preference memory** — remember "I hate layovers", "vegetarian", etc.
- **Trade-off negotiation** — "You're $200 over, downgrade hotel or cut an activity?"
- **Weather-aware** — pull forecasts and adjust outdoor plans
- **Group trip mode** — multiple people input preferences, agent finds the compromise
- **"What if" mode** — "What if I had $500 more?" and it re-optimizes
